In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [21]:
!nvidia-smi

Thu Oct  8 20:27:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             28W /   70W |    7765MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [23]:
!pip install -q -U \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    datasets \
    sentencepiece \
    safetensors \
    huggingface_hub

In [24]:
import transformers
import peft
import accelerate
import bitsandbytes
import datasets
import huggingface_hub

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

Transformers: 5.19.0
PEFT: 0.21.2
Accelerate: 1.15.0
Datasets: 5.1.0
Hugging Face Hub: 1.33.0


In [25]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("Hugging Face authentication completed.")

Hugging Face authentication completed.


In [26]:
from huggingface_hub import model_info

model_name = "NCAIR1/N-ATLaS"

info = model_info(model_name)

print("Model:", info.id)
print("Pipeline:", info.pipeline_tag)

Model: NCAIR1/N-ATLaS
Pipeline: None


In [27]:
# N-ATLAS Loading smoke test
from transformers import AutoTokenizer, AutoConfig

model_name = "NCAIR1/N-ATLaS"

config = AutoConfig.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Model architecture:", config.model_type)
print("Tokenizer:", tokenizer.__class__.__name__)

Model architecture: llama
Tokenizer: TokenizersBackend


In [28]:
# testing tokenizer
text = "Nigeria is a country in West Africa."

tokens = tokenizer(text, return_tensors="pt")

print(tokens)
print("Input shape:", tokens["input_ids"].shape)

{'input_ids': tensor([[128000,     45,  64817,    374,    264,   3224,    304,   4410,  10384,
             13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Input shape: torch.Size([1, 10])


In [29]:
# Load model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)

print("Model loaded successfully.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded successfully.


In [30]:
# Inference Soke Test

prompt = "Nigeria is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
    )

generated = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
)

print(generated)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Nigeria is a country in West Africa with a population of over 200 million people. It is a federal republic with 36 states and one federal capital. The


In [ ]:
## FOCUS: HOW WELL WE  CAN ADAPT N-ATLAS FOR IGALA..

In [31]:
print("Model class:", model.__class__.__name__)
print("Model type:", model.config.model_type)
print("Hidden size:", model.config.hidden_size)
print("Number of layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)
print("Vocabulary size:", model.config.vocab_size)

print("\nTokenizer:")
print("Vocab size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)
print("BOS token:", tokenizer.bos_token)
print("BOS token ID:", tokenizer.bos_token_id)
print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)for name, module in model.named_modules():
    if any(x in name for x in ["q_proj", "k_proj", "v_proj", "o_proj"]):
        print(name)

Model class: LlamaForCausalLM
Model type: llama
Hidden size: 4096
Number of layers: 32
Attention heads: 32
Vocabulary size: 128256

Tokenizer:
Vocab size: 128000
Model max length: 131072
EOS token: <|eot_id|>
EOS token ID: 128009
BOS token: <|begin_of_text|>
BOS token ID: 128000
PAD token: <|eot_id|>
PAD token ID: 128009


In [ ]:
# Inspect actual models name

In [33]:
for name, module in model.named_modules():
    if any(x in name for x in ["q_proj", "k_proj", "v_proj", "o_proj"]):
        print(name)

model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.self_attn.o_proj
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.self_attn.o_proj
model.layers.5.self_attn.q_proj
model.layers.5.self_attn.k_proj
model.layers.5.self_attn.v_proj
model.layers.5.self_attn.o_proj
model.layers.6.self_attn.q_proj
model.layers.6.self_attn.k_proj
model.layers.6.self_attn.v_proj
model.layers.6.self_attn.o_proj
model.layers.7.self_attn.q_proj
model.layers.7.self_attn.k_proj
model.layers.7.self_attn.v_proj
model.la

In [34]:
print(model.config)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.19.0",
  "use_cache": false,
  "vocab_size": 128256
}

